In [14]:
# Read the train dataset
import pandas as pd 

train_df = pd.read_csv('../../data/moral_data_circumstances/dyadic/train_dyadic.tsv', sep='\t')

scenarios = train_df['id'].unique().tolist()

items = []
for id in scenarios:
    rows = train_df[train_df['id'] == id].iloc[:1]
    for i, (_, row) in enumerate(rows.iterrows()):
        human_score = row['human_score_normalized [-4,4]']
        items.append({
            'id': str(row['id']),
            'dataset': row['dataset'],
            'scenario': row['scenario'],
            'human_score': human_score,
        })



In [15]:
len(items)

96

In [16]:
# Randomly pick n items from the list

import random
random.seed(42)
n = 10
picked_items = random.sample(items, n)

# Print the picked items
for item in picked_items:
    print(item)

{'id': '81', 'dataset': 'Effron', 'scenario': 'Ate all the food before anyone else could have any', 'human_score': -2.05333328}
{'id': '14', 'dataset': 'Mickelberg', 'scenario': "Person X presented their colleague's idea as their own in order to get a promotion.", 'human_score': -2.86}
{'id': '3', 'dataset': 'Effron', 'scenario': 'Gave a beggar a five-dollar bill at Christmas time', 'human_score': 1.93103448}
{'id': '94', 'dataset': 'Effron', 'scenario': "Pulled his dog's tail", 'human_score': -2.03428576}
{'id': '35', 'dataset': 'Clifford', 'scenario': 'You see someone cheating in a card game while playing with a group of strangers.', 'human_score': -2.6}
{'id': '31', 'dataset': 'Kruepke', 'scenario': "About a year ago I started re-selling medications from a Canadian pharmacy. I helped a great deal of elderly people so that they didn't have to pay exorbitant amounts for their medications that they needed.", 'human_score': 0.57}
{'id': '28', 'dataset': 'Kruepke', 'scenario': "My husban

In [17]:
# Call gpt to create a reason for the scores

from openai import OpenAI
import os
import json
client = OpenAI(api_key=os.environ["OPENAI_UMASS_API_KEY"])

prompt_template = """
Consider the following scenario: {scenario}.

Now, humans are answering the question: On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?
On an average, humans give a score of {human_score} to this scenario.

Now, you have to come up with the reason or explanation that humans might think of to justify the score.

Your output format should be:
{{
    "explanation": "... (maximum 20 words)",
}}
"""

for item in picked_items:
    prompt = prompt_template.format(scenario=item['scenario'], human_score=item['human_score'])

    print(prompt)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=256,
        temperature=0.0
    )

    generated_str = response.choices[0].message.content
    print(generated_str)
    # Parse the generated string into a dictionary
    # Ignore the ```json and ``` tags
    generated_str = generated_str.replace('```json', '').replace('```', '')
    generated_dict = json.loads(generated_str)
    item['explanation'] = generated_dict['explanation']




Consider the following scenario: Ate all the food before anyone else could have any.

Now, humans are answering the question: On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?
On an average, humans give a score of -2.05333328 to this scenario.

Now, you have to come up with the reason or explanation that humans might think of to justify the score.

Your output format should be:
{
    "explanation": "... (maximum 20 words)",
}

```json
{
    "explanation": "Selfish behavior deprives others of food, showing lack of consideration and fairness."
}
```

Consider the following scenario: Person X presented their colleague's idea as their own in order to get a promotion..

Now, humans are answering the question: On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?
On an average, humans give a score of -2.86 to this scenario.

Now, you have to come up with the reason o

In [18]:
for item in picked_items:
    # print the scenario, human score, and explanation
    print(item['scenario'])
    print(item['human_score'])
    print(item['explanation'])
    print()


Ate all the food before anyone else could have any
-2.05333328
Selfish behavior deprives others of food, showing lack of consideration and fairness.

Person X presented their colleague's idea as their own in order to get a promotion.
-2.86
Stealing credit undermines trust, fairness, and integrity, harming workplace relationships and ethical standards.

Gave a beggar a five-dollar bill at Christmas time
1.93103448
Giving to those in need is good, but impact is limited and context matters.

Pulled his dog's tail
-2.03428576
Pulling a dog's tail causes unnecessary pain and distress, showing a lack of empathy and respect.

You see someone cheating in a card game while playing with a group of strangers.
-2.6
Cheating undermines trust and fairness, harming the integrity of the game and social interaction.

About a year ago I started re-selling medications from a Canadian pharmacy. I helped a great deal of elderly people so that they didn't have to pay exorbitant amounts for their medications

In [19]:
# Create the few shot prompt

few_shot_prompt = ""

few_shot_example_template = """
Question:
Consider a "scenario": {scenario}

On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?

Answer:
{{{{
    "explanation": "{explanation}",
    "moral acceptability score": "{human_score}"
}}}}
"""

few_shot_question_template = """
Question:
Consider a "scenario": {scenario}

On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?

Answer:

"""

few__shot_prompt = ""
for item in picked_items:
    few_shot_prompt += few_shot_example_template.format(
        scenario=item['scenario'],
        explanation=item['explanation'],
        human_score=str(round(item['human_score'], 2))
    )
    few_shot_prompt += "\n\n\n"

few_shot_prompt += few_shot_question_template

print(few_shot_prompt)




Question:
Consider a "scenario": Ate all the food before anyone else could have any

On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?

Answer:
{{
    "explanation": "Selfish behavior deprives others of food, showing lack of consideration and fairness.",
    "moral acceptability score": "-2.05"
}}




Question:
Consider a "scenario": Person X presented their colleague's idea as their own in order to get a promotion.

On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?

Answer:
{{
    "explanation": "Stealing credit undermines trust, fairness, and integrity, harming workplace relationships and ethical standards.",
    "moral acceptability score": "-2.86"
}}




Question:
Consider a "scenario": Gave a beggar a five-dollar bill at Christmas time

On a scale of -4 to 4, how morally acceptable is this scenario (-4 means extremely immoral, 4 means highly moral)?

A